### Project Pipeline
- Project Definition : **Real-Time Emotion Detection**
- Data Collection : **Taken From Kaggle**
- Data Preprocessing : **Resize, Flip, Rotate, Noramalize**
- Model Type : **Vision Transformer (ViT)**
- Evaluation : **Only Accuracy**
- Saving Model : **state-dict**
- Api for Deployment : **FastAPI**
- Testing : **Interact with Streamlit**

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, auc

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), # this required because i train my model on grayscale image
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) 
])

In [36]:
full_dataset = datasets.ImageFolder("dataset/train", transform=transform)
len(full_dataset)

49779

In [37]:
train_size = int(len(full_dataset)*0.75)
test_size = len(full_dataset) - train_size
test_size

12445

In [38]:
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
len(train_dataset)

37334

In [53]:
#data loader
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)
num_classes = len(full_dataset.classes)
print(full_dataset.classes)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [ ]:
#ViT architecture start from here
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x


In [41]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, seq_len):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len + 1, embed_dim))  # Adjusted for [CLS] token

    def forward(self, x):
        return x + self.pos_embed


In [42]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)

    def forward(self, x):
        x = x.transpose(0, 1)   # → [N, B, E]
        x, _ = self.attn(x, x, x)
        x = x.transpose(0, 1) # → [B, N, E]
        return x


In [43]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: [B, N, E]
        x_norm = self.norm1(x)
        x_norm = x_norm.transpose(0, 1)       # [N, B, E]
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        attn_out = attn_out.transpose(0, 1)   # [B, N, E]
        x = x + self.dropout1(attn_out)

        x = x + self.mlp(self.norm2(x))
        return x


In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=10, embed_dim=768, num_heads=8, depth=6, mlp_dim=1024):
        super().__init__()
        self.patch_embedding = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim, (img_size // patch_size) ** 2)
        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim, dropout=0.1) for _ in range(depth)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.norm = nn.LayerNorm(embed_dim)   
        self.mlp_head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embedding(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_encoding(x)
        for block in self.transformer_blocks:
            x = block(x)
        x = self.norm(x)
        return self.mlp_head(x[:, 0])
    
#ViT architectue end here


In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = VisionTransformer(img_size=96, patch_size=4, num_classes=num_classes, embed_dim=128, num_heads=4, depth=4, mlp_dim=360)

model.to(device)

VisionTransformer(
  (patch_embedding): PatchEmbedding(
    (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
  )
  (pos_encoding): PositionalEncoding()
  (transformer_blocks): ModuleList(
    (0-3): 4 x TransformerEncoderBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (dropout1): Dropout(p=0.1, inplace=False)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=128, out_features=360, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=360, out_features=128, bias=True)
        (4): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (mlp_head): Linear(in_features=128, out_features=7, bias=True)
)

In [46]:
from torch.optim.lr_scheduler import CosineAnnealingLR

#traininig loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


def train_step(model, loader):
    model.train()

    running_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss/len(loader)
    
    return avg_loss


In [47]:
epochs = 25
scheduler = CosineAnnealingLR(optimizer, T_max=20)
for epoch in range(epochs):
    loss = train_step(model, train_loader)
    scheduler.step()
    print(f"Epoch: {epoch+1}/{epochs}, Loss: {loss:.4f}")
    if loss <= 0.7:
        break
    torch.cuda.empty_cache()

Epoch: 1/25, Loss: 1.8815
Epoch: 2/25, Loss: 1.7600
Epoch: 3/25, Loss: 1.6987
Epoch: 4/25, Loss: 1.6633
Epoch: 5/25, Loss: 1.6355
Epoch: 6/25, Loss: 1.6135
Epoch: 7/25, Loss: 1.5916
Epoch: 8/25, Loss: 1.5744
Epoch: 9/25, Loss: 1.5575
Epoch: 10/25, Loss: 1.5409
Epoch: 11/25, Loss: 1.5306
Epoch: 12/25, Loss: 1.5185
Epoch: 13/25, Loss: 1.5090
Epoch: 14/25, Loss: 1.5029
Epoch: 15/25, Loss: 1.4966
Epoch: 16/25, Loss: 1.4913
Epoch: 17/25, Loss: 1.4840
Epoch: 18/25, Loss: 1.4813
Epoch: 19/25, Loss: 1.4767
Epoch: 20/25, Loss: 1.4801
Epoch: 21/25, Loss: 1.4788
Epoch: 22/25, Loss: 1.4752
Epoch: 23/25, Loss: 1.4792
Epoch: 24/25, Loss: 1.4789
Epoch: 25/25, Loss: 1.4765


In [ ]:
#torch.save(model.state_dict(), "emotion_classifier.pth")

In [50]:
def evaluate(model, loader):
    model.eval()

    total = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
            total += labels.shape[0]
            correct = correct + (preds == labels).sum().item()

        print(f"Accuracy : {correct/total}")

    # return all_preds, all_labels


In [51]:
evaluate(model, test_loader)

Accuracy : 0.4484531940538369


In [52]:
evaluate(model, train_loader)

Accuracy : 0.45781325333476186
